## Reading 36-2 - Shared Clocks and Torn Reads

By the end of this reading, you will learn:
<ol>
    <li>What an interrupt is, and why it makes sharing data dangerous</li>
    <li>How reading a 64-bit clock in two halves can make time jump backwards by 49.7 days</li>
    <li>How to read a shared counter safely, within a fixed bound</li>
    <li>Why <code>volatile</code> does not make code safe, and what does</li>
</ol>

### Interrupts

A spacecraft's processor does not run one program from top to bottom. When a hardware timer fires, the processor <b>interrupts</b> whatever it is doing, runs a short function called an <b>interrupt service routine (ISR)</b>, and then returns to exactly where it left off. The interrupted code never knows it happened.

That is fine until the ISR and the interrupted code share data. The ISR can run between <i>any</i> two machine instructions, including in the middle of a calculation that reads or writes the shared data.

### A Clock in Two Halves - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec36/torn_read.c">torn_read.c</a>

A 32-bit millisecond counter wraps after 49.7 days, so flight software often keeps a 64-bit count instead. On a 32-bit processor, a 64-bit value is stored as two 32-bit halves. The timer ISR updates both:

    static volatile uint32_t ticks_hi = 0;
    static volatile uint32_t ticks_lo = 0;

    void timer_isr(void)
    {
    	ticks_lo = ticks_lo + 1U;
    	if (ticks_lo == 0U) {
    		ticks_hi = ticks_hi + 1U;
    	}
    }

The rest of the program reads the time by loading the two halves and combining them. This is the code from Ama's slide:

    uint64_t now_unsafe(void)
    {
    	uint32_t hi = ticks_hi;
    	interrupt_point();
    	uint32_t lo = ticks_lo;
    	return ((uint64_t)hi << 32) | lo;
    }

On real hardware, the interrupt could arrive at any moment, and we could not reproduce the problem on demand. So the example <b>simulates</b> the interrupt: <code>interrupt_point()</code> runs the ISR exactly between the two loads. Placing a fault at the worst possible moment is a form of the fault injection you used in Lectures 34 and 35.

Suppose the clock reads 4,294,967,295 ms, one millisecond before the low half wraps:

    $ make torn_read
    Time before the read:   4294967295 ms
    now_unsafe() returned:  0 ms
      TIME WENT BACKWARDS by 49.7 days

The function read the high half (0), then the interrupt carried the count over to high = 1 and low = 0, then the function read the low half (0). It combined the <i>old</i> high half with the <i>new</i> low half. This is called a <b>torn read</b>: the value was read in two pieces, and the pieces came from two different moments.

A test would only catch this if the interrupt happened to land between those two instructions at the exact moment the low half wrapped, which is once every 49.7 days. This is why Ama calls these "interleavings that testing rarely hits."

### Reading a Shared Counter Safely

The fix is to read the high half, read the low half, and then check whether the high half changed in the meantime. If it did, an interrupt carried into it, so read again:

    int now_safe(uint64_t *now)
    {
    	for (int attempt = 0; attempt < MAX_TRIES; attempt++) {
    		uint32_t hi = ticks_hi;
    		interrupt_point();
    		uint32_t lo = ticks_lo;
    		if (hi == ticks_hi) {
    			*now = ((uint64_t)hi << 32) | lo;
    			return 0;
    		}
    	}
    	return -1;
    }

    now_safe() returned:    4294967296 ms

The first attempt saw the high half change, so it tried again and got the correct time.

Ama's slide writes this as a <code>do { ... } while (hi != ticks_hi);</code> loop. That loop is correct, but it has no fixed bound, so it violates Power of Ten Rule 2. In practice the second attempt always succeeds, because the high half changes only once every 49.7 days. But "in practice" is not a proof. Our version stops after <code>MAX_TRIES</code> attempts and returns an error code, so a broken timer becomes a reported anomaly instead of a hang.

Notice also that <code>ticks_hi</code> and <code>ticks_lo</code> are file-scope variables, which Cobra flags under Rule 6. Data shared with an ISR <i>must</i> be visible to both the ISR and the code that reads it, so this is a case where a documented deviation, like the one you wrote in Lecture 35, is the correct answer.

### <code>volatile</code> Is Not Enough

Both halves are declared <code>volatile</code>, the keyword we used in Reading 33-1 for the epsilon experiment. It tells the compiler that the value can change at any time, so every read must actually load it from memory. Without it, the compiler could load <code>ticks_hi</code> once and reuse the value, and our re-read check would never see the change.

But <code>volatile</code> does <b>not</b> make a read or update happen as one indivisible step. <code>now_unsafe</code> uses <code>volatile</code> correctly and is still broken. The tools that actually solve this problem are:
<ul>
    <li><b>Atomic types.</b> C11 added <code>_Atomic</code> and the <code>&lt;stdatomic.h&gt;</code> header. An atomic read or write happens as a single, indivisible operation.</li>
    <li><b>Briefly disabling interrupts</b> while reading the shared data, common on small embedded processors</li>
    <li><b>Message passing</b> instead of shared data: the ISR places the value in a queue that the main program reads, so they never touch the same variable at the same time</li>
</ul>

Torn reads are one kind of <b>concurrency</b> bug. Mars Pathfinder's 1997 resets, from the failure table in Lecture 32, were another: a low-priority task held a lock that a high-priority task needed, and the high-priority task missed its deadline. Engineers fixed it from Earth by turning on a feature of the operating system called <i>priority inheritance</i>. Concurrency bugs are among the hardest to find in testing, which is why flight software teams prefer designs where they cannot happen at all.

### <font color = "red">Class Introduction Question #4 - What is a torn read? Using <code>torn_read.c</code>, explain step by step how <code>now_unsafe()</code> returned a time 49.7 days in the past.</font>

### <font color = "red">Class Introduction Question #5 - How does <code>now_safe()</code> detect a torn read? Why does our version use a <code>for</code> loop with <code>MAX_TRIES</code> instead of the <code>do-while</code> loop on Ama's slide?</font>

### <font color = "red">Class Introduction Question #6 - Both halves of the counter are <code>volatile</code>. Why was that not enough to prevent the torn read, and what would be?</font>

### In class, we will build a mission clock that survives wraparound, rejects stale data, and reads safely while a timer interrupt is running.